# SynthID Text / テキストウォーターマーク検知デモ

このノートブックは SynthID Text とソフトウォーターマークを検知するデモです。Colab では **ランタイム → ランタイムのタイプを変更 → GPU** を推奨します（トークナイザのみなら CPU でも動作します）。

In [ ]:
!git clone https://github.com/kamimai2033/text-sheck.git
%cd text-sheck
!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, '.')
from watermark_detect import detect, SoftWatermarkDetector, SynthIDDetector

## 1. トークナイザの準備

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2')
print(tokenizer.tokenize('これはテキストウォーターマーク検知のデモです。'))

## 2. 通常テキストの検知（ウォーターマークなしの想定）

In [ ]:
normal_text = (
    "The quick brown fox jumps over the lazy dog. " * 20
)
report = detect(normal_text, tokenizer=tokenizer)
print(report)

## 3. ウォーターマーク入りテキストの検知（グリーントークンを偏らせた合成例）

In [ ]:
from watermark_detect.hashing import is_green

vocab_ids = list(range(1000, 3000))
watermarked_ids = [vocab_ids[0]]
for _ in range(400):
    greens = [t for t in vocab_ids if is_green(t, watermarked_ids[-1:])]
    watermarked_ids.append(greens[0] if greens else vocab_ids[-1])
watermarked_text = tokenizer.decode(watermarked_ids)
report = detect(watermarked_text, tokenizer=tokenizer)
print(report)

## 4. SynthID Text の g値列検知

In [ ]:
import random

# 正常時: g値はほぼ一様分布
uniform_g = [random.randrange(4) for _ in range(4000)]
print(SynthIDDetector().detect(uniform_g))

# 水印あり: g値が特定値に偏る
skewed_g = [0] * 3000 + [1] * 300 + [2] * 300 + [3] * 400
print(SynthIDDetector().detect(skewed_g))

## 5. JSON 出力で他システムへ連携

In [ ]:
import json
report = detect(normal_text, tokenizer=tokenizer)
payload = {
    'flagged': report.is_watermarked,
    'soft_watermark': {
        'z_score': report.soft_watermark.z_score,
        'p_value': report.soft_watermark.p_value,
        'green_fraction': report.soft_watermark.green_fraction,
        'num_tokens': report.soft_watermark.num_tokens,
        'is_watermarked': report.soft_watermark.is_watermarked,
    },
}
print(json.dumps(payload, ensure_ascii=False, indent=2))